# Análisis de datos: data_cobre_c2.csv
Este notebook carga y explora los datos químicos y de intensidad de la planta de cobre contenidos en `data_cobre_c2.csv` para posteriormente cargarlos a la base de datos.

In [1]:
import pandas as pd
import psycopg2
import numpy as np

# Configuración de base de datos (utiliza tus credenciales reales)
DB_CONFIG = {
    "dbname": "mydb",
    "user": "myuser",
    "password": "mypassword",
    "host": "localhost",
    "port": 5433
}

# Conectar a la base de datos PostgreSQL
conn = psycopg2.connect(**DB_CONFIG)
cursor = conn.cursor()
print("Conexión a la base de datos establecida.")


Conexión a la base de datos establecida.


In [2]:
# Suponiendo que quieres leer 4 filas empezando desde la fila 4 de datos:
# range(1, 5) saltará las filas 1, 2, 3 y 4 del archivo, pero mantendrá la 0 (el encabezado)
# nrows=4 leerá las siguientes 4 filas después del salto (filas 5, 6, 7 y 8 del archivo original)

#df = pd.read_csv(csv_path, skiprows=range(1, 5), nrows=4)

# Leer el archivo CSV limitando a 1 fila (solo la primera de datos)
csv_path = 'data/raw/data1.csv'
# df = pd.read_csv(csv_path, nrows=4)
_df = pd.read_csv(csv_path)

# Limpiar espacios en los nombres de las columnas
_df.columns = _df.columns.str.strip()

# Limpiar espacios en los strings de MUESTRA
_df['MUESTRA'] = _df['MUESTRA'].astype(str).str.strip()

# Formatear la hora y fecha
_df['time_hhmm'] = pd.to_datetime(_df['time'].astype(str).str.replace('.', ':'), format='mixed').dt.strftime('%H:%M')
_df['date'] = pd.to_datetime(_df['date'], format='mixed').dt.strftime('%Y-%m-%d')

# Manejo de valores nulos o vacíos ("-" a None)
_df = _df.replace({np.nan: None, '-': None})

# df = _df[_df['date']=='2022-12-07']
row = _df[_df['date'].str.contains(r'2022-12-28', regex=True)]
row



,date,time,tara,tweight,dweight,pweight,chemical_id,MUESTRA,pFe,pCu,pZn,pMo,pIns,pSol,time_hhmm
35,2022-12-28,09:52:00,109.5,1160.,312.0,1050.5,4918.0,Concentrado Primario L2,23.14,18.34,0.176,1.08,27.020,29.70,09:52
36,2022-12-28,09:54:00,112.0,1296.5,474.5,1184.5,4914.0,Rebose Hidrociclones L1,2.112,0.478,0.004,0.032,0,40.06,09:54
37,2022-12-28,09:56:00,109.0,1015.,230.5,906.0,4917.0,Concentrado Colectivo,26.84,27.01,0.261,1.56,10.520,25.44,09:56
38,2022-12-28,09:59:00,109.0,1319.5,516.5,1210.5,4915.0,Rebose Hidrociclones L2,2.129,0.568,0.004,0.037,0,42.67,09:59
39,2022-12-28,10:01:00,111.5,1086.,172.5,974.5,4919.0,Concentrado Primario L1,22.73,17.15,0.176,1.03,29.040,17.70,10:01
40,2022-12-28,10:02:00,110.0,1226.,346.5,1116.0,4916.0,Relave Final,1.688,0.059,0.0,0.008,0.000,31.05,10:02


In [ ]:
# Extraer datos clave
csv_date = row1['date']
csv_time = row1['time_hhmm']
csv_muestra = str(row1['MUESTRA']).strip()

ids_permitidos = (17, 18, 19, 20, 21, 22, 23, 24, 25, 26)

# Buscar en la tabla works4cdp_assay
query_buscar_fecha_hora = "SELECT id, sample_id FROM works4cdp_assay WHERE date = %s AND to_char(time, 'HH24:MI') = %s AND sample_id IN %s;"
cursor.execute(query_buscar_fecha_hora, (csv_date, csv_time, ids_permitidos))

ensayos_encontrados = cursor.fetchall()
print(f"Ensayos encontrados a las {csv_time} del {csv_date}: {ensayos_encontrados}")


In [ ]:
fue_actualizado = False

for assay_id, sample_id in ensayos_encontrados:
    
    # Validar nombre en works4cdp_sample
    query_buscar_muestra = "SELECT name, tag FROM works4cdp_sample WHERE id = %s;"
    cursor.execute(query_buscar_muestra, (sample_id,))
    datos_muestra_bd = cursor.fetchone()
    
    if datos_muestra_bd:
        db_muestra_nombre = str(datos_muestra_bd[0]).strip()
        
        if db_muestra_nombre == csv_muestra:
            print(f"¡Coincidencia de Muestra encontrada! Ejecutando UPDATE para ID Assay: {assay_id}...")
            
            update_query = """
                UPDATE works4cdp_assay
                SET 
                    tara = %s, tweight = %s, dweight = %s, pweight = %s,
                    chemical_id = %s, "pFe" = %s, "pCu" = %s, "pZn" = %s,
                    "pMo" = %s, "pIns" = %s, "pSol" = %s
                WHERE id = %s;
            """
            
            # --- SOLUCIÓN AL ERROR numpy.int64 / numpy.float64 ---
            # Convertimos explícitamente a tipos nativos de Python usando int() y float()
            # (Agregamos validación por si algún dato fuera nulo 'None')
            valores_actualizacion = (
                float(row['tara']) if row['tara'] is not None else None,
                float(row['tweight']) if row['tweight'] is not None else None,
                float(row['dweight']) if row['dweight'] is not None else None,
                float(row['pweight']) if row['pweight'] is not None else None,
                int(row['chemical_id']) if row['chemical_id'] is not None else None,
                float(row['pFe']) if row['pFe'] is not None else None,
                float(row['pCu']) if row['pCu'] is not None else None,
                float(row['pZn']) if row['pZn'] is not None else None,
                float(row['pMo']) if row['pMo'] is not None else None,
                float(row['pIns']) if row['pIns'] is not None else None,
                float(row['pSol']) if row['pSol'] is not None else None,
                int(assay_id)
            )
            
            try:
                # Ejecutamos con los tipos nativos
                cursor.execute(update_query, valores_actualizacion)  
                conn.commit()  
                
                print("✅ Fila actualizada exitosamente en la base de datos.")
                fue_actualizado = True
                break
                
            except Exception as e:
                print(f"❌ Error al actualizar la base de datos: {e}")
                conn.rollback()

if not fue_actualizado:
    print("⚠️ No se logró actualizar el ensayo.")

# Cerramos la conexión para liberar recursos
cursor.close()
conn.close()
print("Conexión cerrada.")


In [3]:
ids_permitidos = (17, 18, 19, 20, 21, 22, 23, 24, 25, 26)

for index, row in _df.iterrows():
    csv_date = row['date']
    csv_time = row['time_hhmm']
    csv_muestra = str(row['MUESTRA']).strip()

    query_buscar_fecha_hora = "SELECT id, sample_id FROM works4cdp_assay WHERE date = %s AND to_char(time, 'HH24:MI') = %s AND sample_id IN %s;"
    cursor.execute(query_buscar_fecha_hora, (csv_date, csv_time, ids_permitidos))

    ensayos_encontrados = cursor.fetchall()
    print(f"--- Evaluando Fila {index} ---")
    print(f"Ensayos encontrados a las {csv_time} del {csv_date}: {ensayos_encontrados}")

    fue_actualizado = False

    for assay_id, sample_id in ensayos_encontrados:
        
        query_buscar_muestra = "SELECT name, tag FROM works4cdp_sample WHERE id = %s;"
        cursor.execute(query_buscar_muestra, (sample_id,))
        datos_muestra_bd = cursor.fetchone()
        
        if datos_muestra_bd:
            db_muestra_nombre = str(datos_muestra_bd[0]).strip()
            
            if db_muestra_nombre == csv_muestra:
                print(f"¡Coincidencia de Muestra encontrada! Ejecutando UPDATE para ID Assay: {assay_id}...")
                
                update_query = """
                    UPDATE works4cdp_assay
                    SET 
                        tara = %s, tweight = %s, dweight = %s, pweight = %s,
                        chemical_id = %s, "pFe" = %s, "pCu" = %s, "pZn" = %s,
                        "pMo" = %s, "pIns" = %s, "pSol" = %s
                    WHERE id = %s;
                """
                
                valores_actualizacion = (
                    float(row['tara']) if row['tara'] is not None else None,
                    float(row['tweight']) if row['tweight'] is not None else None,
                    float(row['dweight']) if row['dweight'] is not None else None,
                    float(row['pweight']) if row['pweight'] is not None else None,
                    int(row['chemical_id']) if row['chemical_id'] is not None else None,
                    float(row['pFe']) if row['pFe'] is not None else None,
                    float(row['pCu']) if row['pCu'] is not None else None,
                    float(row['pZn']) if row['pZn'] is not None else None,
                    float(row['pMo']) if row['pMo'] is not None else None,
                    float(row['pIns']) if row['pIns'] is not None else None,
                    float(row['pSol']) if row['pSol'] is not None else None,
                    int(assay_id)
                )
                
                try:
                    cursor.execute(update_query, valores_actualizacion)  
                    conn.commit()  
                    
                    print("✅ Fila actualizada exitosamente en la base de datos.\n")
                    fue_actualizado = True
                    break
                    
                except Exception as e:
                    print(f"❌ Error al actualizar la base de datos: {e}\n")
                    conn.rollback()

    if not fue_actualizado:
        print("⚠️ No se logró actualizar el ensayo.\n")

# Cerramos la conexión para liberar recursos
cursor.close()
conn.close()
print("Conexión cerrada.")


--- Evaluando Fila 0 ---
Ensayos encontrados a las 15:23 del 2022-12-05: [(13058, 19)]
¡Coincidencia de Muestra encontrada! Ejecutando UPDATE para ID Assay: 13058...
✅ Fila actualizada exitosamente en la base de datos.

--- Evaluando Fila 1 ---
Ensayos encontrados a las 15:25 del 2022-12-05: [(12667, 26)]
¡Coincidencia de Muestra encontrada! Ejecutando UPDATE para ID Assay: 12667...
✅ Fila actualizada exitosamente en la base de datos.

--- Evaluando Fila 2 ---
Ensayos encontrados a las 15:27 del 2022-12-05: [(10231, 17)]
¡Coincidencia de Muestra encontrada! Ejecutando UPDATE para ID Assay: 10231...
✅ Fila actualizada exitosamente en la base de datos.

--- Evaluando Fila 3 ---
Ensayos encontrados a las 15:38 del 2022-12-05: [(12283, 24)]
¡Coincidencia de Muestra encontrada! Ejecutando UPDATE para ID Assay: 12283...
✅ Fila actualizada exitosamente en la base de datos.

--- Evaluando Fila 4 ---
Ensayos encontrados a las 12:04 del 2022-12-07: [(10230, 17)]
¡Coincidencia de Muestra encontra

ValueError: could not convert string to float: '  -   '